In [1]:
library(Gviz)
library(GenomicRanges)
library(Rsamtools)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

In [2]:
BiocManager::version()        # Bioconductor version
packageVersion("Gviz")        # Gviz package version specifically

[1] '3.18'

[1] '1.46.1'

In [3]:
library(Gviz)
library(GenomicAlignments)
library(rtracklayer)

bam_file <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/mainAmpliconicGeneReads_gencode.bam"   # needs a .bai index alongside it
gtf_file <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"  # RefSeq GTF for CHM13

chr <- "NC_060948.1"
genome_name <- "hs1"  # arbitrary label for CHM13 — Gviz just uses it for display

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: 'MatrixGenerics'


The following objects are masked from 'package:matrixStats':

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges

In [4]:
# Replace with your gene's coordinates on NC_060948.1
locus_start <- 25568883
locus_end   <- 25643428

# DAZ

### DAZ1

In [29]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup + repeats using Gviz
# CHM13 T2T assembly — no GTF required
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3"

CHROM       <- "HG01358_chrY"
START       <- 23658771
END         <- 23752464

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ1_gencodeReads.pdf"
PLOT_WIDTH  <- 14
PLOT_HEIGHT <- 8

# ---- 2. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 3. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 4. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format = "gff3",
  which  = GRanges(CHROM, IRanges(START, END))
)

# repeat_class column contains e.g. "SINE/Alu", "LINE/L1", "DNA/TcMar-Tigger"
# Collapse to broad family by stripping everything after "/"
repeat_class <- as.character(mcols(repeats_gr)$repeat_class)
broad_class  <- sub("/.*", "", repeat_class)
broad_class[is.na(broad_class) | broad_class == ""] <- "Unknown"

message("Broad classes found: ", paste(sort(unique(broad_class)), collapse = ", "))

# ---- Colour palette ----------------------------------------
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# ---- Build AnnotationTrack ---------------------------------
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

all_classes <- unique(broad_class)
colour_list <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 5. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads & Repeats \u2014 ", CHROM, ":",
                       format(START, big.mark = ","), "\u2013",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3, 0.8)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam

Loading repeat annotations from /project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3

Broad classes found: DNA, LINE, Low_complexity, LTR, Simple_repeat, SINE, tRNA

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ1_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:23,658,771–23,752,464' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:23,658,771–23,752,464' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & R

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ1_gencodeReads.pdf



In [ ]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup + repeats using Gviz
# CHM13 T2T assembly — no GTF required
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG00731chrY_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG00731_chrY.gff3"

CHROM       <- "HG00731_chrY"
START       <- 23871493   
END         <- 24044881

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ1_gencodeReads.pdf"
PLOT_WIDTH  <- 14
PLOT_HEIGHT <- 8

# ---- 2. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 3. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 4. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format = "gff3",
  which  = GRanges(CHROM, IRanges(START, END))
)

# repeat_class column contains e.g. "SINE/Alu", "LINE/L1", "DNA/TcMar-Tigger"
# Collapse to broad family by stripping everything after "/"
repeat_class <- as.character(mcols(repeats_gr)$repeat_class)
broad_class  <- sub("/.*", "", repeat_class)
broad_class[is.na(broad_class) | broad_class == ""] <- "Unknown"

message("Broad classes found: ", paste(sort(unique(broad_class)), collapse = ", "))

# ---- Colour palette ----------------------------------------
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# ---- Build AnnotationTrack ---------------------------------
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

all_classes <- unique(broad_class)
colour_list <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 5. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads & Repeats \u2014 ", CHROM, ":",
                       format(START, big.mark = ","), "\u2013",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3, 0.8)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

In [8]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGenes_gencode_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 23932690
END         <- 24035860

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ1_gencode.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGen

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ1_gencode.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


### DAZ2

In [30]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup + repeats using Gviz
# CHM13 T2T assembly — no GTF required
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3"

CHROM       <- "HG01358_chrY"
START       <- 23769691   
END         <- 23857814         

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ2_gencodeReads.pdf"
PLOT_WIDTH  <- 14
PLOT_HEIGHT <- 8

# ---- 2. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 3. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 4. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format = "gff3",
  which  = GRanges(CHROM, IRanges(START, END))
)

# repeat_class column contains e.g. "SINE/Alu", "LINE/L1", "DNA/TcMar-Tigger"
# Collapse to broad family by stripping everything after "/"
repeat_class <- as.character(mcols(repeats_gr)$repeat_class)
broad_class  <- sub("/.*", "", repeat_class)
broad_class[is.na(broad_class) | broad_class == ""] <- "Unknown"

message("Broad classes found: ", paste(sort(unique(broad_class)), collapse = ", "))

# ---- Colour palette ----------------------------------------
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# ---- Build AnnotationTrack ---------------------------------
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

all_classes <- unique(broad_class)
colour_list <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 5. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads & Repeats \u2014 ", CHROM, ":",
                       format(START, big.mark = ","), "\u2013",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3, 0.8)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam

Loading repeat annotations from /project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3

Broad classes found: DNA, LINE, Low_complexity, Simple_repeat, SINE, tRNA

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ2_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:23,769,691–23,857,814' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:23,769,691–23,857,814' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeat

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ2_gencodeReads.pdf



In [11]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})
options(ucscChromosomeNames = FALSE)

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG00731chrY_splicingNoDupes.bam"

# Region to plot
CHROM       <- "HG00731_chrY"
START       <- 24064161      
END         <- 24152993

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ2_gencodeReads.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12


# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index <your_bam>.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track (commented out — uncomment to re-enable) -------------
# To re-enable:
#   1. Uncomment REPEAT_FILE in section 1
#   2. Uncomment this entire section
#   3. Add rptrack back to plotTracks() list in section 7
#   4. Restore sizes = c(0.5, 3, 0.8, 1.5) in section 7

# REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# palette_cols <- c(
#   "SINE"           = "#FF0000",
#   "LINE"           = "#FFA500",
#   "DNA"            = "#0101FF",
#   "LTR"            = "#2EFF69",
#   "Satellite"      = "#F4B9FF",
#   "Simple_repeat"  = "#A52B2B",
#   "Low_complexity" = "#A52B2B",
#   "rRNA"           = "#000000",
#   "tRNA"           = "#000000",
#   "snRNA"          = "#000000",
#   "RNA"            = "#000000",
#   "Unknown"        = "#000000"
# )
#
# message("Loading repeat annotations from ", REPEAT_FILE)
# repeats_gr <- import(
#   REPEAT_FILE,
#   format     = "gff3",
#   which      = GRanges(CHROM, IRanges(START, END))
# )
#
# target_vals  <- as.character(mcols(repeats_gr)$Target)
# repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
# repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"
# broad_class  <- sub("/.*", "", repeat_class)
#
# feat_colours <- ifelse(broad_class %in% names(palette_cols),
#                        palette_cols[broad_class],
#                        "#BDBDBD")
#
# rptrack <- AnnotationTrack(
#   repeats_gr,
#   chromosome       = CHROM,
#   name             = "RepeatMasker",
#   feature          = broad_class,
#   stacking         = "dense",
#   fontsize         = 7,
#   background.title = "#7B3294",
#   col.title        = "white",
#   col              = NA
# )
#
# all_classes <- unique(broad_class)
# colour_list <- setNames(
#   ifelse(all_classes %in% names(palette_cols),
#          palette_cols[all_classes],
#          "#BDBDBD"),
#   all_classes
# )
# displayPars(rptrack) <- as.list(colour_list)


# ---- 6. RefSeq transcript track (commented out — uncomment to re-enable) ---
# To re-enable:
#   1. Uncomment GTF_FILE in section 1
#   2. Uncomment this entire section
#   3. Add grtrack back to plotTracks() list in section 7
#   4. Restore sizes = c(0.5, 3, 1.5) -> c(0.5, 3, 0.8, 1.5) if repeats also on

# GTF_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"

# message("Building TxDb from GTF: ", GTF_FILE)
# txdb <- makeTxDbFromGFF(
#   GTF_FILE,
#   format = "gtf",
#   organism = "Homo sapiens"
# )
#
# message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
# grtrack <- GeneRegionTrack(
#   txdb,
#   chromosome           = CHROM,
#   start                = START,
#   end                  = END,
#   name                 = "RefSeq\nTranscripts",
#   transcriptAnnotation = "symbol",
#   collapseTranscripts  = FALSE,
#   fill                 = "#4292C6",
#   col                  = "#084594",
#   fontcolor.group      = "black",
#   fontsize             = 9,
#   background.title     = "#2171B5",
#   col.title            = "white"
# )


# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)


# ---- 8. (Optional) list all chromosomes in the TxDb --------
# Uncomment once grtrack/txdb are re-enabled:
# message("\nAvailable chromosomes / sequences in this GTF:")
# print(seqlevels(txdb))

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG00731chrY_splicingNoDupes.bam

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ2_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:24,064,161–24,152,993' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:24,064,161–24,152,993' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:24,064,161–24,152,993' in 'mbcsToSbcs': dot substituted for <94>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:24,064,161–

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ2_gencodeReads.pdf



In [12]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGenes_gencode_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 24042764
END         <- 24104456

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ2_daz4_gencode.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGen

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ2_daz4_gencode.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


### DAZ4

In [35]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup + repeats using Gviz
# CHM13 T2T assembly — no GTF required
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3"

CHROM       <- "HG01358_chrY"
START       <- 25309797         
END         <- 25392665         

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ4_gencodeReads.pdf"
PLOT_WIDTH  <- 14
PLOT_HEIGHT <- 8

# ---- 2. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 3. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 4. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format = "gff3",
  which  = GRanges(CHROM, IRanges(START, END))
)

# repeat_class column contains e.g. "SINE/Alu", "LINE/L1", "DNA/TcMar-Tigger"
# Collapse to broad family by stripping everything after "/"
repeat_class <- as.character(mcols(repeats_gr)$repeat_class)
broad_class  <- sub("/.*", "", repeat_class)
broad_class[is.na(broad_class) | broad_class == ""] <- "Unknown"

message("Broad classes found: ", paste(sort(unique(broad_class)), collapse = ", "))

# ---- Colour palette ----------------------------------------
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# ---- Build AnnotationTrack ---------------------------------
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

all_classes <- unique(broad_class)
colour_list <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 5. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads & Repeats \u2014 ", CHROM, ":",
                       format(START, big.mark = ","), "\u2013",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3, 0.8)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam

Loading repeat annotations from /project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3

Broad classes found: DNA, LINE, Low_complexity, LTR, Simple_repeat, SINE, tRNA

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ4_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:25,309,797–25,392,665' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:25,309,797–25,392,665' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & R

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ4_gencodeReads.pdf



In [14]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})
options(ucscChromosomeNames = FALSE)

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG00731chrY_splicingNoDupes.bam"

# Region to plot
CHROM       <- "HG00731_chrY"
START       <- 25591829         
END         <- 25671564

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ4_gencodeReads.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12


# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index <your_bam>.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track (commented out — uncomment to re-enable) -------------
# To re-enable:
#   1. Uncomment REPEAT_FILE in section 1
#   2. Uncomment this entire section
#   3. Add rptrack back to plotTracks() list in section 7
#   4. Restore sizes = c(0.5, 3, 0.8, 1.5) in section 7

# REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# palette_cols <- c(
#   "SINE"           = "#FF0000",
#   "LINE"           = "#FFA500",
#   "DNA"            = "#0101FF",
#   "LTR"            = "#2EFF69",
#   "Satellite"      = "#F4B9FF",
#   "Simple_repeat"  = "#A52B2B",
#   "Low_complexity" = "#A52B2B",
#   "rRNA"           = "#000000",
#   "tRNA"           = "#000000",
#   "snRNA"          = "#000000",
#   "RNA"            = "#000000",
#   "Unknown"        = "#000000"
# )
#
# message("Loading repeat annotations from ", REPEAT_FILE)
# repeats_gr <- import(
#   REPEAT_FILE,
#   format     = "gff3",
#   which      = GRanges(CHROM, IRanges(START, END))
# )
#
# target_vals  <- as.character(mcols(repeats_gr)$Target)
# repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
# repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"
# broad_class  <- sub("/.*", "", repeat_class)
#
# feat_colours <- ifelse(broad_class %in% names(palette_cols),
#                        palette_cols[broad_class],
#                        "#BDBDBD")
#
# rptrack <- AnnotationTrack(
#   repeats_gr,
#   chromosome       = CHROM,
#   name             = "RepeatMasker",
#   feature          = broad_class,
#   stacking         = "dense",
#   fontsize         = 7,
#   background.title = "#7B3294",
#   col.title        = "white",
#   col              = NA
# )
#
# all_classes <- unique(broad_class)
# colour_list <- setNames(
#   ifelse(all_classes %in% names(palette_cols),
#          palette_cols[all_classes],
#          "#BDBDBD"),
#   all_classes
# )
# displayPars(rptrack) <- as.list(colour_list)


# ---- 6. RefSeq transcript track (commented out — uncomment to re-enable) ---
# To re-enable:
#   1. Uncomment GTF_FILE in section 1
#   2. Uncomment this entire section
#   3. Add grtrack back to plotTracks() list in section 7
#   4. Restore sizes = c(0.5, 3, 1.5) -> c(0.5, 3, 0.8, 1.5) if repeats also on

# GTF_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"

# message("Building TxDb from GTF: ", GTF_FILE)
# txdb <- makeTxDbFromGFF(
#   GTF_FILE,
#   format = "gtf",
#   organism = "Homo sapiens"
# )
#
# message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
# grtrack <- GeneRegionTrack(
#   txdb,
#   chromosome           = CHROM,
#   start                = START,
#   end                  = END,
#   name                 = "RefSeq\nTranscripts",
#   transcriptAnnotation = "symbol",
#   collapseTranscripts  = FALSE,
#   fill                 = "#4292C6",
#   col                  = "#084594",
#   fontcolor.group      = "black",
#   fontsize             = 9,
#   background.title     = "#2171B5",
#   col.title            = "white"
# )


# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)


# ---- 8. (Optional) list all chromosomes in the TxDb --------
# Uncomment once grtrack/txdb are re-enabled:
# message("\nAvailable chromosomes / sequences in this GTF:")
# print(seqlevels(txdb))

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG00731chrY_splicingNoDupes.bam

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ4_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,591,829–25,671,564' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,591,829–25,671,564' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,591,829–25,671,564' in 'mbcsToSbcs': dot substituted for <94>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,591,829–

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ4_gencodeReads.pdf



In [15]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGenes_gencode_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 25547208
END         <- 25645628

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ4_daz2_gencode.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGen

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ4_daz2_gencode.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


### DAZ3


In [34]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup + repeats using Gviz
# CHM13 T2T assembly — no GTF required
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3"

CHROM       <- "HG01358_chrY"
START       <- 25409071         
END         <- 25487757         

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ3_gencodeReads.pdf"
PLOT_WIDTH  <- 14
PLOT_HEIGHT <- 8

# ---- 2. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 3. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 4. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format = "gff3",
  which  = GRanges(CHROM, IRanges(START, END))
)

# repeat_class column contains e.g. "SINE/Alu", "LINE/L1", "DNA/TcMar-Tigger"
# Collapse to broad family by stripping everything after "/"
repeat_class <- as.character(mcols(repeats_gr)$repeat_class)
broad_class  <- sub("/.*", "", repeat_class)
broad_class[is.na(broad_class) | broad_class == ""] <- "Unknown"

message("Broad classes found: ", paste(sort(unique(broad_class)), collapse = ", "))

# ---- Colour palette ----------------------------------------
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# ---- Build AnnotationTrack ---------------------------------
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

all_classes <- unique(broad_class)
colour_list <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 5. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads & Repeats \u2014 ", CHROM, ":",
                       format(START, big.mark = ","), "\u2013",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3, 0.8)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG01358chrY_splicingNoDupes.bam

Loading repeat annotations from /project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/HG01358_chrY.gff3

Broad classes found: DNA, LINE, Low_complexity, Simple_repeat, SINE, tRNA

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ3_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:25,409,071–25,487,757' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — HG01358_chrY:25,409,071–25,487,757' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeat

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG01358_DAZ3_gencodeReads.pdf



In [17]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})
options(ucscChromosomeNames = FALSE)

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG00731chrY_splicingNoDupes.bam"

# Region to plot
CHROM       <- "HG00731_chrY"
START       <- 25688792         
END         <- 25772864

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ3_gencodeReads.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12


# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index <your_bam>.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(HG01358)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track (commented out — uncomment to re-enable) -------------
# To re-enable:
#   1. Uncomment REPEAT_FILE in section 1
#   2. Uncomment this entire section
#   3. Add rptrack back to plotTracks() list in section 7
#   4. Restore sizes = c(0.5, 3, 0.8, 1.5) in section 7

# REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# palette_cols <- c(
#   "SINE"           = "#FF0000",
#   "LINE"           = "#FFA500",
#   "DNA"            = "#0101FF",
#   "LTR"            = "#2EFF69",
#   "Satellite"      = "#F4B9FF",
#   "Simple_repeat"  = "#A52B2B",
#   "Low_complexity" = "#A52B2B",
#   "rRNA"           = "#000000",
#   "tRNA"           = "#000000",
#   "snRNA"          = "#000000",
#   "RNA"            = "#000000",
#   "Unknown"        = "#000000"
# )
#
# message("Loading repeat annotations from ", REPEAT_FILE)
# repeats_gr <- import(
#   REPEAT_FILE,
#   format     = "gff3",
#   which      = GRanges(CHROM, IRanges(START, END))
# )
#
# target_vals  <- as.character(mcols(repeats_gr)$Target)
# repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
# repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"
# broad_class  <- sub("/.*", "", repeat_class)
#
# feat_colours <- ifelse(broad_class %in% names(palette_cols),
#                        palette_cols[broad_class],
#                        "#BDBDBD")
#
# rptrack <- AnnotationTrack(
#   repeats_gr,
#   chromosome       = CHROM,
#   name             = "RepeatMasker",
#   feature          = broad_class,
#   stacking         = "dense",
#   fontsize         = 7,
#   background.title = "#7B3294",
#   col.title        = "white",
#   col              = NA
# )
#
# all_classes <- unique(broad_class)
# colour_list <- setNames(
#   ifelse(all_classes %in% names(palette_cols),
#          palette_cols[all_classes],
#          "#BDBDBD"),
#   all_classes
# )
# displayPars(rptrack) <- as.list(colour_list)


# ---- 6. RefSeq transcript track (commented out — uncomment to re-enable) ---
# To re-enable:
#   1. Uncomment GTF_FILE in section 1
#   2. Uncomment this entire section
#   3. Add grtrack back to plotTracks() list in section 7
#   4. Restore sizes = c(0.5, 3, 1.5) -> c(0.5, 3, 0.8, 1.5) if repeats also on

# GTF_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"

# message("Building TxDb from GTF: ", GTF_FILE)
# txdb <- makeTxDbFromGFF(
#   GTF_FILE,
#   format = "gtf",
#   organism = "Homo sapiens"
# )
#
# message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
# grtrack <- GeneRegionTrack(
#   txdb,
#   chromosome           = CHROM,
#   start                = START,
#   end                  = END,
#   name                 = "RefSeq\nTranscripts",
#   transcriptAnnotation = "symbol",
#   collapseTranscripts  = FALSE,
#   fill                 = "#4292C6",
#   col                  = "#084594",
#   fontcolor.group      = "black",
#   fontsize             = 9,
#   background.title     = "#2171B5",
#   col.title            = "white"
# )


# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)


# ---- 8. (Optional) list all chromosomes in the TxDb --------
# Uncomment once grtrack/txdb are re-enabled:
# message("\nAvailable chromosomes / sequences in this GTF:")
# print(seqlevels(txdb))

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_HG00731chrY_splicingNoDupes.bam

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ3_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,688,792–25,772,864' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,688,792–25,772,864' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,688,792–25,772,864' in 'mbcsToSbcs': dot substituted for <94>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads — HG00731_chrY:25,688,792–

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/HG00731_DAZ3_gencodeReads.pdf



In [18]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGenes_gencode_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 25659902
END         <- 25738259

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ3_daz3_gencode.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGen

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ3_daz3_gencode.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


In [21]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGenes_pooledTestis_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 25544178
END         <- 25647348

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ2_pooledTestis.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGen

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ2_pooledTestis.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


In [22]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGenes_pooledTestis_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 25654230
END         <- 25745113

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ3_pooledTestis.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGen

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ3_pooledTestis.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


In [23]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGenes_gencode_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 25654230
END         <- 25745113

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ3_gencode.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/alignments/maleGen

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_DAZ3_gencode.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


# TSPY2

In [4]:
#!/usr/bin/env Rscript
# ============================================================
# Plot BAM pileup + repeats using Gviz
# CHM13 T2T assembly — no GTF required
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})
options(ucscChromosomeNames = FALSE)

# ---- 1. User settings — edit these -------------------------
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_NA20850chrY_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/NA20850_chrY.gff3"

CHROM       <- "NA20850_chrY"
START       <- 5918333         
END         <- 5921128         

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/NA20850_TSPY2_gencodeReads.pdf"
PLOT_WIDTH  <- 14
PLOT_HEIGHT <- 8

# ---- 2. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 3. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(NA20850)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 4. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format = "gff3",
  which  = GRanges(CHROM, IRanges(START, END))
)

# repeat_class column contains e.g. "SINE/Alu", "LINE/L1", "DNA/TcMar-Tigger"
# Collapse to broad family by stripping everything after "/"
repeat_class <- as.character(mcols(repeats_gr)$repeat_class)
broad_class  <- sub("/.*", "", repeat_class)
broad_class[is.na(broad_class) | broad_class == ""] <- "Unknown"

message("Broad classes found: ", paste(sort(unique(broad_class)), collapse = ", "))

# ---- Colour palette ----------------------------------------
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# ---- Build AnnotationTrack ---------------------------------
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

all_classes <- unique(broad_class)
colour_list <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 5. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("Reads & Repeats \u2014 ", CHROM, ":",
                       format(START, big.mark = ","), "\u2013",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  sizes       = c(0.5, 3, 0.8)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/mappedAssembly/maleGeneReads_NA20850chrY_splicingNoDupes.bam

Loading repeat annotations from /project/mkonkel/tangeno/users/giannim/chrY/HG00731_HG01358_repeats/NA20850_chrY.gff3

Broad classes found: 

Plotting to /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/NA20850_TSPY2_gencodeReads.pdf

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — NA20850_chrY:5,918,333–5,921,128' in 'mbcsToSbcs': dot substituted for <e2>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — NA20850_chrY:5,918,333–5,921,128' in 'mbcsToSbcs': dot substituted for <80>"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
"conversion failure on 'Reads & Repeats — NA20850_chrY:5,918,333–5,921,128' in 'mbcsToSbcs': 

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/NA20850_TSPY2_gencodeReads.pdf



In [16]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmedMappedCHM13/SRR31360662_trimmed_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 5915859
END         <- 5936792

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_TSPY2_gencode_extended.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmedMappedCHM13/SRR31360662_t

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_TSPY2_gencode_extended.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


# RBMY

In [17]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmedMappedCHM13/SRR31360662_trimmed_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 22704545
END         <- 22730990

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_RBMY1D_gencode.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmedMappedCHM13/SRR31360662_t

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_RBMY1D_gencode.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


In [18]:
#!/usr/bin/env Rscript
# ============================================================
# Plot RefSeq transcripts + BAM pileup + repeats using Gviz
# GTF: GCF_009914755.1/genomic.gtf  (CHM13 T2T assembly)
# ============================================================

# ---- 0. Install / load packages ----------------------------
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

for (pkg in c("Gviz", "GenomicFeatures", "rtracklayer", "IRanges", "GenomeInfoDb")) {
  if (!requireNamespace(pkg, quietly = TRUE))
    BiocManager::install(pkg, ask = FALSE)
}

suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicFeatures)
  library(rtracklayer)
  library(IRanges)
  library(GenomeInfoDb)
})

# ---- 1. User settings — edit these -------------------------
GTF_FILE    <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic.gtf"
BAM_FILE    <- "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmedMappedCHM13/SRR31360662_trimmed_splicingNoDupes.bam"
REPEAT_FILE <- "/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/hs1_chrY.gff3"

# Region to plot
CHROM       <- "NC_060948.1"
START       <- 22704298
END         <- 22732703

OUTPUT_PDF  <- "/project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_RBMY1D_gencode.pdf"
PLOT_WIDTH  <- 14   # inches
PLOT_HEIGHT <- 12

# ---- 2. Build a TxDb from the GTF --------------------------
message("Building TxDb from GTF (this may take a minute)...")
txdb <- makeTxDbFromGFF(
  file       = GTF_FILE,
  format     = "gtf",
  dataSource = "NCBI RefSeq",
  organism   = "Homo sapiens"
)

# ---- 3. Genome axis track ----------------------------------
gaxis <- GenomeAxisTrack(
  name     = CHROM,
  col      = "black",
  fontsize = 9
)

# ---- 4. BAM pileup track -----------------------------------
message("Loading BAM pileup track from ", BAM_FILE)
# NOTE: BAM must be indexed — run: samtools index SRR12544673.bam
altrack <- AlignmentsTrack(
  BAM_FILE,
  chromosome       = CHROM,
  start            = START,
  end              = END,
  name             = "Reads\n(SRR12544673)",
  type             = "pileup",
  col.coverage     = "#238B45",
  fill.coverage    = "#74C476",
  background.title = "#006D2C",
  col.title        = "white",
  fontsize         = 9
)

# ---- 5. Repeats track --------------------------------------
message("Loading repeat annotations from ", REPEAT_FILE)
repeats_gr <- import(
  REPEAT_FILE,
  format     = "gff3",
  which      = GRanges(CHROM, IRanges(START, END))
)

# The GFF3 Target attribute format is:
#   Target=<repeat_name> <repeat_class> <start> <end>
# Extract the repeat class (2nd word of Target).
target_vals  <- as.character(mcols(repeats_gr)$Target)
repeat_class <- sub("^\\S+\\s+(\\S+).*", "\\1", target_vals)
repeat_class[is.na(repeat_class) | repeat_class == ""] <- "Unknown"

# Derive the broad repeat family from the full class string
# e.g. "SINE/Alu" -> "SINE", "LINE/L1" -> "LINE", "DNA/hAT-Charlie" -> "DNA"
broad_class <- sub("/.*", "", repeat_class)

# Colour palette keyed on broad class
palette_cols <- c(
  "SINE"           = "#FF0000",
  "LINE"           = "#FFA500",
  "DNA"            = "#0101FF",
  "LTR"            = "#2EFF69",
  "Satellite"      = "#F4B9FF",
  "Simple_repeat"  = "#A52B2B",
  "Low_complexity" = "#A52B2B",
  "rRNA"           = "#000000",
  "tRNA"           = "#000000",
  "snRNA"          = "#000000",
  "RNA"            = "#000000",
  "Unknown"        = "#000000"
)

# Map each element to a colour via its broad class
feat_colours <- ifelse(broad_class %in% names(palette_cols),
                       palette_cols[broad_class],
                       "#BDBDBD")

# Build the AnnotationTrack — use broad_class as the feature label
# so Gviz grouping/legend reflects meaningful categories
rptrack <- AnnotationTrack(
  repeats_gr,
  chromosome       = CHROM,
  name             = "RepeatMasker",
  feature          = broad_class,
  stacking         = "dense",
  fontsize         = 7,
  background.title = "#7B3294",
  col.title        = "white",
  col              = NA
)

# Apply colours using the correct assignment form: displayPars<-(track) <- list(...)
# This is the proper Gviz API — do.call(displayPars, ...) does NOT work for setting.
all_classes  <- unique(broad_class)
colour_list  <- setNames(
  ifelse(all_classes %in% names(palette_cols),
         palette_cols[all_classes],
         "#BDBDBD"),
  all_classes
)
displayPars(rptrack) <- as.list(colour_list)

# ---- 6. RefSeq transcript track ----------------------------
message("Creating GeneRegionTrack for ", CHROM, ":", START, "-", END)
grtrack <- GeneRegionTrack(
  txdb,
  chromosome           = CHROM,
  start                = START,
  end                  = END,
  name                 = "RefSeq\nTranscripts",
  transcriptAnnotation = "symbol",
  collapseTranscripts  = FALSE,
  fill                 = "#4292C6",
  col                  = "#084594",
  fontcolor.group      = "black",
  fontsize             = 9,
  background.title     = "#2171B5",
  col.title            = "white"
)

# ---- 7. Plot -----------------------------------------------
message("Plotting to ", OUTPUT_PDF)
pdf(OUTPUT_PDF, width = PLOT_WIDTH, height = PLOT_HEIGHT)

plotTracks(
  list(gaxis, altrack, rptrack, grtrack),
  chromosome  = CHROM,
  from        = START,
  to          = END,
  main        = paste0("RefSeq Transcripts, Reads & Repeats — ", CHROM, ":",
                       format(START, big.mark = ","), "–",
                       format(END,   big.mark = ",")),
  cex.main    = 1.1,
  title.width = 1.6,
  # Relative track heights: axis | reads | repeats | transcripts
  sizes       = c(0.5, 3, 0.8, 1.5)
)

dev.off()
message("Done! Output saved to: ", OUTPUT_PDF)

# ---- 8. (Optional) list all chromosomes in the GTF ---------
message("\nAvailable chromosomes / sequences in this GTF:")
print(seqlevels(txdb))

Building TxDb from GTF (this may take a minute)...

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
"The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored."
Warning message in .find_exon_cds(exons, cds):
"The following transcripts have exons that contain more than one CDS
  (only the first CDS was kept for each exon): NM_001134939.1,
  NM_001172437.2, NM_001184961.1, NM_001301020.1, NM_001301302.1,
  NM_001301371.1, NM_002537.3, NM_004152.3, NM_015068.3, NM_016178.2"
Warning message in .reject_transcripts(bad_tx, because):
"The following transcripts were dropped because they have incompatible
  CDS and stop codons: NM_001172437.2, NM_001184961.1, NM_015068.3"
OK

Loading BAM pileup track from /project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmedMappedCHM13/SRR31360662_t

pdf 
  2

Done! Output saved to: /project/mkonkel/tangeno/users/giannim/chrY/expressionFigureAlignments/figures/T2T_RBMY1D_gencode.pdf


Available chromosomes / sequences in this GTF:



 [1] "NC_060925.1" "NC_060926.1" "NC_060927.1" "NC_060928.1" "NC_060929.1"
 [6] "NC_060930.1" "NC_060931.1" "NC_060932.1" "NC_060933.1" "NC_060934.1"
[11] "NC_060935.1" "NC_060936.1" "NC_060937.1" "NC_060938.1" "NC_060939.1"
[16] "NC_060940.1" "NC_060941.1" "NC_060942.1" "NC_060943.1" "NC_060944.1"
[21] "NC_060945.1" "NC_060946.1" "NC_060947.1" "NC_060948.1"


# Assembly